In [1]:
import os
import warnings
import itertools
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime
import time
from scipy import stats
timestr = time.strftime("%Y%m%d-%H%M%S")
import joblib

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import ndcg_score, make_scorer


# Modeling


In [3]:
from google.colab import drive
drive.mount('/content/drive')
# list files in the folder
print(os.listdir("/content/drive/MyDrive/DMT_data"))

ModuleNotFoundError: No module named 'google.colab'

In [3]:
!ln -s "/content/drive/MyDrive/DMT_data" /content/DMT_data
# This should now show your data
!ls /content/DMT_data

ln: failed to create symbolic link '/content/DMT_data/DMT_data': File exists
DMT_data  test_set_VU_DM.csv  training_set_VU_DM.csv


Plot style

In [2]:
plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.color":        "#e4e4e4",
    "grid.linewidth":    0.05,
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.titleweight":  "bold",
})

BLUE   = "#378ADD"
TEAL   = "#1D9E75"
AMBER  = "#EF9F27"
CORAL  = "#D85A30"
PURPLE = "#7F77DD"
GRAY   = "#888780"

## Read data

In [3]:
data_path = '/content/DMT_data'
data_path = '../data'
train_file = os.path.join(data_path, 'training_set_VU_DM.csv')
test_file = os.path.join(data_path, 'test_set_VU_DM.csv')

train_df = pd.read_csv(train_file)
df = train_df.copy()
display(df.head())
#test = pd.read_csv(test_file)

,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,893,3,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
1,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,10404,4,4.0,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
2,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,21315,3,4.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0
3,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,27348,2,4.0,...,NaN,NaN,NaN,NaN,-1.0,0.0,5.0,0,NaN,0
4,1,2013-04-04 08:32:15,12,187,NaN,NaN,219,29604,4,3.5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,0,NaN,0


# needs to be changed later: a function with all preprecessing steps we did

In [ ]:
def preprocessing(data):
    df = data.copy()
    # preprocessing of target varibales (which will not be present in the test file) - code will run only if the column exists
    if 'booking_bool' in data.columns:
        # ADD: potential cleaning functions we want here
        pass

    # impute visitor history star rating with a 10% trimmed mean
    if 'visitor_hist_starrating' in df.columns:
        trimmed_mean = stats.trim_mean(df['visitor_hist_starrating'].dropna(), 0.1)
        df['visitor_hist_starrating'] = df['visitor_hist_starrating'].fillna(trimmed_mean)

    # competitor summary features
    rate_cols = [f'comp{i}_rate' for i in range(1, 9)]
    if all(col in df.columns for col in rate_cols):
        #df['comp_score_sum'] = df[rate_cols].sum(axis=1, skipna=True)
        df['is_cheapest_any'] = (df[rate_cols] == 1).any(axis=1).astype(int)
        df['is_expensive_any'] = (df[rate_cols] == -1).any(axis=1).astype(int)

    # drop the raw competitor columns if they exist.
    comp_cols_to_drop = [
        *rate_cols,
        *[f'comp{i}_inv' for i in range(1, 9)],
        *[f'comp{i}_rate_percent_diff' for i in range(1, 9)],
    ]
    df = df.drop(columns=[col for col in comp_cols_to_drop if col in df.columns], errors='ignore')

    # remove leakage column
    if 'gross_bookings_usd' in df.columns:
        df = df.drop(columns=['gross_bookings_usd'])

    # search affinity flag and imputation
    if 'srch_query_affinity_score' in df.columns:
        df['has_query_affinity'] = df['srch_query_affinity_score'].notnull().astype(int)
        df['srch_query_affinity_score'] = df['srch_query_affinity_score'].fillna(0)

    # fill sparse location score with 0.
    if 'prop_location_score2' in df.columns:
        df['prop_location_score2'] = df['prop_location_score2'].fillna(0)

    # median imputation
    median_cols = ['visitor_hist_adr_usd','orig_destination_distance','prop_review_score',]
    median_cols = [col for col in median_cols if col in df.columns]
    if median_cols:
        df[median_cols] = df[median_cols].fillna(df[median_cols].median())

    df["diff_starrating"] = df["visitor_hist_starrating"] - df["prop_starrating"]

    return df
df = preprocessing(df)

Read to see what are the options:
https://towardsdatascience.com/a-complete-guide-to-recommender-system-tutorial-with-sklearn-surprise-keras-recommender-5e52e8ceace1/

## K-Nearest Neighbours
https://www.geeksforgeeks.org/machine-learning/recommender-systems-using-knn/


Content-Based Recommender System - but we can add use features and make it hybrid

In [5]:
from sklearn.model_selection import  GroupKFold # group stratification so data from the entire search stay together
features = [ 'visitor_hist_starrating', 'prop_starrating', 'prop_review_score','prop_location_score1', 'prop_location_score2',
    'srch_length_of_stay', 'srch_booking_window']
target = 'booking_bool'

# sample to make CV feasible
df_sample = df.sample(n=50000, random_state=42) 
X = df_sample[features + ['srch_id']] #we keep srch_id out of the trianing columsn so that they are not used as a feature
y = df_sample[target]

# train test split based on the search_id
unique_ids = X['srch_id'].unique()
train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)

X_train_full = X[X['srch_id'].isin(train_ids)]
y_train = y[X['srch_id'].isin(train_ids)]
X_test_holdout = X[X['srch_id'].isin(test_ids)]
y_test_holdout = y[X['srch_id'].isin(test_ids)]

# save groups for CV and drop 'srch_id' from features
train_groups = X_train_full['srch_id']
X_train = X_train_full[features]
X_test_final = X_test_holdout[features]

# 3. Define the Scorer
def ndcg_scorer(y, y_pred, **kwargs):
    y_true = [[1 if x == y.iloc[i] else 0 for x in np.unique(y)] for i in range(len(y))]
    return ndcg_score(y_true, y_pred, **kwargs)

scorer = make_scorer(ndcg_scorer, needs_proba=True, k=5)

# 4. pipeline - so that scaling is recalculated for every fold in CV
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# param grid
param_grid = {
    'knn__n_neighbors': [10, 20, 50],
    'knn__weights': ['distance'],
    'knn__metric': ['manhattan']
}

#Cross-Validation- stratifiedKFold because booking_bool is highly imbalanced
cv_strategy = GroupKFold(n_splits=5)

grid_search = GridSearchCV(
    pipeline, 
    param_grid=param_grid, 
    cv=cv_strategy, 
    scoring=scorer, 
    n_jobs=-1,
    verbose=2
)

print("Starting Cross-Validation Grid Search...")
grid_search.fit(X_train, y_train, groups=train_groups)

print(f"Best CV Score ({scorer}): {grid_search.best_score_:.4f}")
print(f"Best Parameters: {grid_search.best_params_}")

Starting Cross-Validation Grid Search...
Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV] END knn__metric=manhattan, knn__n_neighbors=10, knn__weights=distance; total time=   2.6s
[CV] END knn__metric=manhattan, knn__n_neighbors=10, knn__weights=distance; total time=   2.7s
[CV] END knn__metric=manhattan, knn__n_neighbors=10, knn__weights=distance; total time=   2.7s
[CV] END knn__metric=manhattan, knn__n_neighbors=10, knn__weights=distance; total time=   2.7s
[CV] END knn__metric=manhattan, knn__n_neighbors=10, knn__weights=distance; total time=   2.8s
[CV] END knn__metric=manhattan, knn__n_neighbors=20, knn__weights=distance; total time=   2.9s
[CV] END knn__metric=manhattan, knn__n_neighbors=20, knn__weights=distance; total time=   3.0s
[CV] END knn__metric=manhattan, knn__n_neighbors=20, knn__weights=distance; total time=   3.2s
[CV] END knn__metric=manhattan, knn__n_neighbors=20, knn__weights=distance; total time=   3.2s
[CV] END knn__metric=manhattan, knn__n_neigh

In [7]:
# re-fit the best estimator on a larger training slice
best_model = grid_search.best_estimator_
best_model.fit(df[features], df[target]) # fit on the full train dataset

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('knn', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",3

In [10]:
#1. load and prepare the data
data_path = '/content/DMT_data'
data_path = '../data'
test_file_df = pd.read_csv(os.path.join(data_path, 'test_set_VU_DM.csv'))

#apply the pre-processing function
test_file_df = preprocessing(test_file_df)

columns_to_process = features + ['srch_id', 'prop_id']
# select same features used in training and srch_id and prop_id for identification
X_test_file = test_file_df[columns_to_process].copy()

# final application to the test set
X_test_file['booking_probability'] = best_model.predict_proba(X_test_file[features])[:, 1]

# sort by SearchID and the booking probability
submission = X_test_file.sort_values(['srch_id', 'booking_probability'], ascending=[True, False])


In [13]:
display(submission.head(15))
len(submission)

,visitor_hist_starrating,prop_starrating,prop_review_score,prop_location_score1,prop_location_score2,srch_length_of_stay,srch_booking_window,srch_id,prop_id,booking_probability
24,3.381305,3,4.5,2.83,0.2303,1,10,1,123675,0.189139
4,3.381305,3,4.5,2.94,0.2090,1,10,1,24194,0.162299
9,3.381305,3,4.0,2.08,0.1649,1,10,1,54937,0.104166
20,3.381305,3,4.5,2.77,0.0569,1,10,1,90385,0.101444
23,3.381305,3,4.0,2.40,0.2182,1,10,1,99484,0.097252
3,3.381305,3,4.5,2.40,0.0561,1,10,1,22393,0.092976
6,3.381305,3,4.5,3.09,0.1300,1,10,1,34263,0.088689
15,3.381305,3,4.5,2.89,0.2272,1,10,1,73666,0.085819
13,3.381305,3,4.5,2.77,0.2261,1,10,1,63894,0.076770
16,3.381305,3,4.0,2.77,0.2003,1,10,1,74045,0.074333


4959183

In [12]:
submission_df = submission[['srch_id', 'prop_id']].copy()

#save
submission_df.to_csv(f'../output/VU-DM-2026-Group-62-{timestr}.csv', index=False)
print(f"File of length {len(submission)} saved")

File of length 4959183 saved


In [ ]:

submission[['srch_id', 'prop_id']].to_csv('submission_knn.csv', index=False)

In [19]:
def save_model_bundle(model, scaler, folder='models'):
    # create directory if it doesn't exist
    
    date_str = datetime.now().strftime("%Y%m%d")  #timestamped filename
    
    # save each component - joblib is more efficient for large files
    joblib.dump(model, f'{folder}/knn_{date_str}.pkl')
    joblib.dump(scaler, f'{folder}/scaler_knn_{date_str}.pkl')
    
    print(f"Model and scaler saved in /{folder} with date {date_str}")

best_pipeline = grid_search.best_estimator_
fitted_scaler = best_pipeline.named_steps['scaler']
fitted_knn = best_pipeline.named_steps['knn']

save_model_bundle(fitted_knn, fitted_scaler)

Model and scaler saved in /models with date 20260508


# K-Means

In [ ]:
# 1. feature selection - for now only a few features to see how the model performs
features = [
    'prop_starrating', 'prop_review_score', 'prop_location_score1',
    'prop_location_score2', 'prop_log_historical_price',
    'price_usd', 'promotion_flag'
]

# 2. Preprocessing Pipeline
# extract features and handle missing values/scaling
X = df[features].copy()


#scaling
scaler = StandardScaler() #scaling
X_scaled = scaler.fit_transform(X)

# 3. train-test split (80-20)
# training set to find centroids; test set to evaluate assignment stability
X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)


# Elbow method to find the best k
inertia = []
K_range = range(1, 30)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

# Plotting the Elbow
plt.figure(figsize=(8, 4))
plt.plot(K_range, inertia, 'bx-')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method For Optimal k')
plt.show()

In [ ]:

# 4. unsupervised Cross-Validation (inertia stability)
# we use K-Fold to ensure the clusters (inertia) are consistent across different slices of data
kf = KFold(n_splits=5, shuffle=True, random_state=42)
k_clusters = 30 #change for the optimal
cv_inertia = []

print(f"Starting 5-Fold Cross-Validation for K={k_clusters}...")

for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train[train_index], X_train[val_index]

    # fit model on the fold training data
    km_fold = KMeans(n_clusters=k_clusters, n_init=5, random_state=42)
    km_fold.fit(X_fold_train)

    # calculate Inertia on the validation fold
    # inertia = sum of squared distances of samples to their closest cluster center
    cv_inertia.append(km_fold.inertia_)

print(f"Average CV Inertia: {np.mean(cv_inertia):.2f} (+/- {np.std(cv_inertia):.2f})")


In [ ]:
# 1. feature selection - for now only a few features to see how the model performs
features = [
    'prop_starrating', 'prop_review_score', 'prop_location_score1',
    'prop_location_score2', 'prop_log_historical_price',
    'price_usd', 'promotion_flag'
]

# 2. Preprocessing Pipeline
# extract features and handle missing values/scaling
X = df[features].copy()  #TO DO: add more features we want to model

# data imputation
imputer = SimpleImputer(strategy='median') #missing values with median
X_processed = imputer.fit_transform(X)

#scaling
scaler = StandardScaler() #scaling
X_scaled = scaler.fit_transform(X_processed)

# 3. train-test split (80-20)
# training set to find centroids; test set to evaluate assignment stability
X_train, X_test = train_test_split(X_scaled, test_size=0.2, random_state=42)

# 4. unsupervised Cross-Validation (inertia stability)
# we use K-Fold to ensure the clusters (inertia) are consistent across different slices of data
kf = KFold(n_splits=5, shuffle=True, random_state=42)
k_clusters = 50
cv_inertia = []

print(f"Starting 5-Fold Cross-Validation for K={k_clusters}...")

for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train[train_index], X_train[val_index]

    # fit model on the fold training data
    km_fold = KMeans(n_clusters=k_clusters, n_init=5, random_state=42)
    km_fold.fit(X_fold_train)

    # calculate Inertia on the validation fold
    # inertia = sum of squared distances of samples to their closest cluster center
    cv_inertia.append(km_fold.inertia_)

print(f"Average CV Inertia: {np.mean(cv_inertia):.2f} (+/- {np.std(cv_inertia):.2f})")

Starting 5-Fold Cross-Validation for K=50...
Average CV Inertia: 2647813.22 (+/- 20324.80)


In [7]:
# 5. final model implementation
# fitting on the full training set and assigning clusters to the test set
final_kmeans = KMeans(n_clusters=k_clusters, n_init=10, random_state=42)
final_kmeans.fit(X_train)

# assign cluster labels back to the original dataframe for the test indices
train_indices, test_indices = train_test_split(df.index, test_size=0.2, random_state=42)
df.loc[train_indices, 'hotel_cluster'] = final_kmeans.labels_
df.loc[test_indices, 'hotel_cluster'] = final_kmeans.predict(X_test)

print("\nCluster assignment complete. Sample distribution:")
print(df['hotel_cluster'].value_counts().head())


Cluster assignment complete. Sample distribution:
hotel_cluster
17.0    309776
2.0     306708
40.0    294968
43.0    199867
23.0    198242
Name: count, dtype: int64


#### Computing the predicted ranking from the clusters - pointwise ranking
Goal: "rank the properties belonging to a user search on the likeliness that the property will be booked" for each user list all the properites from their search based on the probability that that property will be booked, i.e. highest probability that booking_bool=1

The evaluation function is (NDCG)@5 calculated per query and averaged over all queries with the values weighted by the log_2 function - this rewards ehavily if the acutal booked hotel appears at the very top of our list.

Since K-means is unsupervised, the "Cluster ID" itself doesn't tell "probability." We have to map the clusters to historical booking success. The way to do this:

1. Calculate the *Mean Booking Rate* for each cluster.

2. Assign this "Cluster Score" to every hotel in that cluster.

3. Sort the hotels within each SearchId by that score.


cluster_score - Probability Estimate derived from historical data. total number of bookings in a cluster/total number of hotel listings which were put in that cluster e.g if Cluster 5 has 1,000 hotel listings and 50 of them were booked, its cluster_score is 0.05.

In [10]:
# visualising the differences between the culsters - average attributes for each cluster
cluster_profiles = df.groupby('hotel_cluster')[features + ['booking_bool']].mean()

print("Cluster profiles by booking probability:")
display(cluster_profiles.sort_values(by='booking_bool', ascending=False).head(10)) # sort by booking_bool - most booked clusters

Cluster Profiles (Sorted by Booking Probability):


,prop_starrating,prop_review_score,prop_location_score1,prop_location_score2,prop_log_historical_price,price_usd,promotion_flag,booking_bool
hotel_cluster,,,,,,,,
3.0,2.853198,3.902936,2.253529,0.580634,4.769558,1.290011e+02,0.009174,0.121240
5.0,4.636364,3.954545,3.462727,0.056500,2.006364,4.285486e+06,0.272727,0.090909
20.0,2.868457,4.050294,2.476815,0.259172,4.830187,1.276556e+02,0.000000,0.069983
49.0,3.659463,4.023662,4.782237,0.649141,5.361562,1.921578e+02,1.000000,0.063155
33.0,3.185057,3.783923,3.794737,0.265938,5.007226,1.299185e+02,1.000000,0.061267
36.0,3.639277,4.010657,4.618811,0.448703,0.000000,2.109575e+02,1.000000,0.055184
34.0,4.184211,4.263158,3.245526,0.071414,3.907105,2.552001e+06,0.368421,0.052632
26.0,3.513439,4.074922,4.210516,0.467427,0.000000,2.457806e+02,0.000000,0.049329
18.0,3.962934,4.156162,4.793386,0.375484,5.458065,2.247765e+02,1.000000,0.044785


In [ ]:
# 1. calculate the cluster_score of each cluster based on historical bookings
cluster_weights = df.groupby('hotel_cluster')['booking_bool'].mean().reset_index()
cluster_weights.columns = ['hotel_cluster', 'cluster_score']

# 2. merge this score back to into the df
final_df = df.merge(cluster_weights, on='hotel_cluster', how='left')

# 3. double sorting to handle tie-breakers
# if two hotels are in the same cluster, they have the same cluster_score.
# we use prop_review_score as a secondary sort to break ties - we can consider sorting first by a differnt feature to break ties (for exmaple the location)
final_df = final_df.sort_values(by=['srch_id', 'cluster_score', 'prop_review_score'], ascending=[True, False, False])


# save file
final_df.to_csv(f'VU-DM-2026-Group-62-{timestr}.csv', index=False)

print("K-means Ranker Submission Generated.")
print(final_df.head(30).to_string(index=False))

### Saving the model, scaler and imputer 
in case sth crashes - then we can recover the model and use for submission

In [ ]:
def save_model_bundle(model, scaler, imputer, weights, folder='models'):
    # create directory if it doesn't exist
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    date_str = datetime.now().strftime("%Y%m%d")  #timestamped filename
    
    # save each component - joblib is more efficient for large files
    joblib.dump(model, f'{folder}/kmeans_{date_str}.pkl')
    joblib.dump(scaler, f'{folder}/scaler_kmeans_{date_str}.pkl')
    joblib.dump(imputer, f'{folder}/imputer_kmeans_{date_str}.pkl')
    joblib.dump(weights, f'{folder}/weights_kmeans_{date_str}.pkl')
    
    print(f"Model, scaler, imputer and cluster weigths saved in /{folder} with date {date_str}")

save_model_bundle(final_kmeans, scaler, imputer, cluster_weights)

### Test file - final prediction of K-means


In [ ]:
#1. load and prepare the data
data_path = '/content/DMT_data'
test_file_df = pd.read_csv(os.path.join(data_path, 'test_set_VU_DM.csv'))

#apply the pre-processing function
#TO DO
test_file_df = preprocessing(test_file_df)

# select same features used in training
X_test_file = test_file_df[features].copy()

# apply the imputer and scaler - the already fitted one (do not re-fit)
X_test_file_imputed = imputer.transform(X_test_file)
X_test_file_scaled = scaler.transform(X_test_file_imputed)

# 2. assign clusters - we use .predict() to see which training cluster these new rows belong to
test_file_df['hotel_cluster'] = final_kmeans.predict(X_test_file_scaled) #final_kmeans is the model we fitted before

# 3. apply the cluster scores we calculated from the training
test_file_df = test_file_df.merge(cluster_weights, on='hotel_cluster', how='left') #cluster_weigths are the probabilities of each cluster

# in case any clsuters in tests were not present in training we fill them with global mean
test_file_df['cluster_score'] = test_file_df['cluster_score'].fillna(df['booking_bool'].mean())

# 4. ouble sorting to handle tie-breakers
test_file_ranked = test_file_df.sort_values(by=['srch_id', 'cluster_score', 'prop_review_score'], ascending=[True, False, False])

test_file_ranked.to_csv(f'VU-DM-2026-Group-62-{timestr}.csv', index=False) #save the submission file

# XGBoost
or a different model

##